# The planning problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/05_problem.ipynb)

`PlanningProblem` is the declarative task specification read by every planner, controller evaluator, and learner—whether deterministic or stochastic, finite- or infinite-horizon, continuous-time or on a clock (`StepPlanningProblem`).

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `PlanningProblem`: what it holds

A `PlanningProblem` states *what* to achieve and under *what* constraints, never how to solve it:

$$\min J = \int_{t_0}^{t_0 + T} e^{-\rho (t - t_0)} g(x, u, t)\, dt + e^{-\rho T} h(x(t_0 + T))
\quad \text{s.t.} \quad \dot x = f(x, u, t),\; x(t) \in X,\; u(t) \in U,\;
x(t_0) \in X_0,\; x(t_0 + T) \in X_f.$$

It is the continuous task: the system, the sets, the cost and the horizon $T$. Step counts and the other discretization choices belong to the solvers.

### Constructor

`PlanningProblem(sys, *, x0=None, cost=None, X=None, U=None, X0=None, Xf=None, horizon=None, t0=0.0, penalize_failure=False, failure_cost=None, discount_rate=0.0, distributions=None, disturbances=None, criterion="expectation")`

Every argument is an attribute of the same name, read back after coercion.

| Argument | Type | Default | Meaning |
| --- | --- | --- | --- |
| `sys` | `System` | required | The plant |
| `x0` | `array (n,)`, `Distribution` or `Set` | from `X0`, else `sys.x0` | The evaluation start: a point, or a law over $x(t_0)$ (a set is the uniform law on it). `problem.x0` is then the point (a law's mean) and `problem.start_law` the law (`None` for a point) |
| `X0` | `Set` or `array (n,)` | the singleton at `x0`, or the support of its law | The start constraint $x(t_0) \in X_0$ that planners enforce |
| `X` | `Set` | unconstrained | State constraint along the horizon; `X=sys.state.box` is the plant's declared range |
| `U` | `InputSet` or `BoxSet` | unconstrained | Input constraint; a `BoxSet` is wrapped in a `BoxInputSet`; `U=sys.input_box` is the ports' declared range |
| `Xf` | `Set` or `array (n,)` | `None`, a free end | Terminal constraint $x(t_0 + T) \in X_f$; a vector is the singleton at it |
| `cost` | `CostFunction` | `None` | The running and terminal cost; `None` for a pure feasibility problem |
| `horizon` | `float` | `None` | The horizon length $T$ in seconds; `None` or `inf` is the infinite horizon |
| `t0` | `float` | `0.0` | The start time $t_0$ |
| `discount_rate` | `float` | `0.0` | The continuous discount rate $\rho \ge 0$ |
| `penalize_failure` | `bool` | `False` | Whether an evaluation adds the failure cost to the $J$ of a run that leaves `X` or `U`, stops being finite, or misses `Xf` |
| `failure_cost` | `float` | `None` | The finite cost of a failure; `None` is `cost_bound()` |
| `distributions` | mapping | `None` | Laws over parameters, in the shape of `problem.params`: `{"plant": {"m": Uniform(...)}}` |
| `disturbances` | `dict[str, System]` | `None` | Source blocks on the plant's input ports: `{"d": WhiteNoise(...)}` |
| `criterion` | `"expectation"`, `"worst_case"` or `("cvar", alpha)` | `"expectation"` | What an evaluation's `objective` reduces the draws' $J$ to |

### Properties and methods

| Call | Returns | Meaning |
| --- | --- | --- |
| `problem.params` | `dict` | The numbers by role: the parts' own trees (`"plant"`, `"cost"`, `"X"`, `"U"`, `"X0"` unless it is the singleton at `x0`, `"Xf"` when set, each disturbance source under its block name) and the problem's own (`"x0"`, `"t0"`, `"discount_rate"`). Assigning a complete tree routes each entry to its part |
| `problem.is_stochastic` | `bool` | Whether a part is random: a start law, a law in `distributions`, a random source |
| `problem.horizon_kind()` | `"finite"` or `"infinite"` | `"infinite"` when `horizon` is `None` or `inf` |
| `problem.goal_point()` | `ndarray (n,)` | The point a heuristic aims at: `Xf`'s point, else the cost's `x_bar` when inside `X`, else the start. It never constrains a plan |
| `problem.discount_factor(dt)` | `float` | The per-step factor $\gamma = e^{-\rho\,\Delta t}$ |
| `problem.cost_bound(horizon=None, *, discount_rate=None, dt=None, params=None)` | `float` | An upper bound of the cost of any run that stays in the boxes of `X` and `U`: $g_{max} \int_0^T e^{-\rho t} dt + e^{-\rho T} h_{max}$ |
| `problem.failure_cost_at(horizon=None, *, discount_rate=None, dt=None, params=None)` | `float` | `failure_cost` when set, else `cost_bound(...)` |
| `problem.replace(**changes)` | `PlanningProblem` | A new problem with the named constructor arguments replaced, the others shared |
| `problem.drawn_paths()` | `list[tuple[str, ...]]` | The paths of the leaves `realize` draws |
| `problem.sample_x0(key, n=None)` | `array (n_x,)`, or `(n, n_x)` with `n` | Starts drawn from the start law (the point itself without one) |
| `problem.sample_params(key)` | `dict` | One draw of each law of `distributions`, at its path |
| `problem.realize(key=None, n=1)` | `dict` | `n` realizations as one tree in the shape of `params`: `"x0"` `(n, n_x)`, each drawn leaf `(n, *shape)`; `key=None` gives the means |
| `problem.nominal()` | `PlanningProblem` | The certainty-equivalent problem: a copy with every law at its mean and every random source at its mean signal |
| `problem.discretize(dt, *, solver="rk4", substeps=1)` | `StepPlanningProblem` | The problem on its plant sampled at `dt` (section 3) |
| `problem.compile(controller, dt=None, *, solver="rk4", substeps=None, backend="auto", episode_length=None)` | `ProblemEvaluator` | The evaluator of a controller on the problem ([`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb)) |

In [ ]:
import numpy as np

from minilink import BallSet, Pendulum, PlanningProblem, QuadraticCost

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(plant, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1), x_bar=UP)

problem = PlanningProblem(
    plant,
    x0=np.array([0.0, 0.0]),
    cost=cost,
    X=plant.state.box,
    U=plant.input_box,
    Xf=BallSet(UP, 0.2),
    horizon=5.0,
)
problem

In [ ]:
list(problem.params), problem.x0, problem.X0, problem.goal_point(), problem.horizon_kind()

In [ ]:
problem.cost_bound(), problem.failure_cost_at()

`problem.replace(**changes)` derives another problem without mutating the first: here the finite-horizon swing-up becomes an infinite-horizon discounted problem.

In [ ]:
infinite_problem = problem.replace(horizon=None, Xf=None, discount_rate=0.5)
(
    infinite_problem.horizon_kind(),
    infinite_problem.discount_factor(0.05),
    infinite_problem.cost_bound(),
)

## 2. Stochastic problems

A problem is stochastic by its parts. Where a number goes, a law may go:

1. `x0=` a `Distribution` (or a `Set`, the uniform law on it); `X0` defaults to its support;
2. `distributions=` laws over parameters, in the tree's own shape;
3. `disturbances=` random source blocks (such as `WhiteNoise`) on the plant's input ports. The source of the port `"d"` is the block `disturbance` of the loop an evaluator closes, the source of `"n"` its `noise`.

In [ ]:
from minilink import Uniform

random_problem = PlanningProblem(
    plant,
    x0=Uniform([-0.2, -0.2], [0.2, 0.2]),
    cost=cost,
    horizon=5.0,
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
(
    random_problem.is_stochastic,
    random_problem.drawn_paths(),
    random_problem.start_law,
    random_problem.X0,
)

In [ ]:
# One start (2,), then three starts (3, 2)
random_problem.sample_x0(0), random_problem.sample_x0(0, n=3)

In [ ]:
draws = random_problem.realize(0, n=3)
draws["x0"].shape, draws["plant"]["m"], random_problem.sample_params(0)

In [ ]:
# The certainty-equivalent problem: every law at its mean
nominal = random_problem.nominal()
nominal.is_stochastic, nominal.x0, nominal.sys.params["m"]

A `WhiteNoise` block on the disturbance port `"d"` of `NoisyPendulum`; `realize` draws one seed per trial for it:

In [ ]:
from minilink import NoisyPendulum, WhiteNoise

noisy_plant = NoisyPendulum()
noisy_cost = QuadraticCost.from_system(noisy_plant, Q=np.eye(2), R=np.diag([1.0, 0.0, 0.0]))

noisy_problem = PlanningProblem(
    noisy_plant,
    x0=np.zeros(2),
    cost=noisy_cost,
    horizon=5.0,
    disturbances={"d": WhiteNoise(1, psd=0.04, sample_period=0.01, seed=1)},
)
list(noisy_plant.inputs), noisy_problem.drawn_paths(), noisy_problem.realize(0, n=3)["disturbance"]

## 3. `StepPlanningProblem`: the problem on a clock

Solvers that work tick by tick (dynamic programming, reinforcement learning, discrete rollouts) read the discrete-time problem

$$x_{k+1} = F(x_k, u_k, k), \qquad J = \sum_{k<N} \gamma^k c(x_k, u_k, k) + \gamma^N h(x_N, N).$$

`problem.discretize(dt, *, solver="rk4", substeps=1)` builds it from a continuous problem: the plant is a `SampledSystem`, the cost a `SampledCost`, $N = T / \Delta t$ and $\gamma = e^{-\rho \Delta t}$. A problem native to a step system is built directly:

`StepPlanningProblem(sys, *, n_steps=None, cost=None, gamma=1.0, x0=None, X=None, U=None, X0=None, Xf=None, t0=0, dt=None, penalize_failure=False, failure_cost=None, distributions=None, criterion="expectation")`

with `sys` a `StepSystem`, `cost` a `StepCost`, and `n_steps=None` the infinite horizon. The step problem answers its own equations:

| Call | Returns | Meaning |
| --- | --- | --- |
| `step_problem.sys`, `step_problem.cost` | `StepSystem`, `StepCost` | The plant on the clock and the stage cost (a `SampledSystem` and a `SampledCost` on a discretized problem) |
| `step_problem.n_steps`, `step_problem.gamma` | `int \| None`, `float` | The horizon in ticks $N$ and the discount per tick $\gamma$ |
| `step_problem.dt`, `step_problem.t0` | `float \| None`, number | The period and the start time in seconds on a discretized problem; on a native problem `dt` is `None` and `t0` the integer tick offset |
| `step_problem.x0`, `step_problem.start_law`, `step_problem.X`, `step_problem.U`, `step_problem.X0`, `step_problem.Xf` | as on `PlanningProblem` | The start and the sets |
| `step_problem.penalize_failure`, `step_problem.failure_cost`, `step_problem.criterion`, `step_problem.distributions`, `step_problem.is_stochastic` | as on `PlanningProblem` | The failure switch, the criterion and the laws |
| `step_problem.params` | `dict` | The numbers by role: `"plant"`, `"cost"`, the sets', `"x0"`, `"t0"`, `"gamma"` |
| `step_problem.transition(x, u, k, params=None)` | `(x_next, c_k)` | Tick $k$: the next state $x_{k+1} = F(x_k, u_k)$ and the stage cost $c_k$ |
| `step_problem.terminal(x, t=None, params=None)` | scalar | The terminal cost $h(x_N)$ at the horizon's end (or at the time `t`); `0` on an infinite horizon |
| `step_problem.rollout(controller, x0=None, params=None, n_steps=None)` | `Trajectory` | The run under `controller`, a controller block or a law `u = policy(x, t)`, over $N$ ticks: `x` `(n, N + 1)`, `u` `(m, N + 1)`, and the signal `"J"` `(1, N + 1)`, $J_k = \sum_{j<k} \gamma^j c_j$ with $\gamma^N h(x_N)$ added at the last sample |
| `step_problem.horizon_kind()` | `"finite"` or `"infinite"` | `"infinite"` when `n_steps` is `None` |
| `step_problem.cost_bound(n_steps=None, *, gamma=None, params=None)` | `float` | $\sum_{k<N} \gamma^k c_{max} + \gamma^N h_{max}$ |
| `step_problem.failure_cost_at(n_steps=None, *, gamma=None, params=None)` | `float` | `failure_cost` when set, else `cost_bound(...)` |
| `step_problem.replace(**changes)`, `step_problem.realize(key=None, n=1)`, `step_problem.sample_x0(key, n=None)`, `step_problem.sample_params(key)` | as on `PlanningProblem` | The same verbs on the step problem's own tree |

In [ ]:
step_problem = problem.discretize(0.05)
(
    type(step_problem.sys).__name__,
    type(step_problem.cost).__name__,
    step_problem.dt,
    step_problem.n_steps,
    step_problem.gamma,
    list(step_problem.params),
)

In [ ]:
x = np.zeros(2)
u = np.array([1.0])
x_next, c_0 = step_problem.transition(x, u, k=0)
x_next, c_0, step_problem.terminal(UP), step_problem.cost_bound()

In [ ]:
# A constant torque as the law u = policy(x, t)
traj = step_problem.rollout(lambda x, t: np.array([2.0]))
traj.x.shape, traj.u.shape, traj.signal("J")[0, -1]

Next, [`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb): `ProblemEvaluator`, `Evaluation`, `compare`, and `sys.compile()`.